# Assignment 3 — BGR Graph Classification: No-Grad Evaluation

This notebook demonstrates how to evaluate a pre-trained **GraphSAGE** model
on a Building-Ground Relationship (BGR) graph using `torch.no_grad()`.

**No training is performed. Model weights are not modified.**

---

## What this notebook does

1. Explains the no-grad evaluation workflow
2. Shows the required CSV input format
3. Loads and verifies the input graph
4. Loads the pre-trained `bgr_model.pt`
5. Runs inference with `model.eval()` and `torch.no_grad()`
6. Displays the predicted BGR class
7. Discusses limitations

## Class Mapping

| Class | Label |
|-------|-------|
| 0 | Separation |
| 1 | Separation with Plinth |
| 2 | Adherence |
| 3 | Adherence with Plinth |
| 4 | Interlock |

## Node Feature Encoding

Each graph node is a spatial cell from the building model.
The 5 features are a **one-hot encoding** of the cell type:

| Feature | Cell type |
|---------|----------|
| feat_0 | ground |
| feat_1 | column |
| feat_2 | plinth |
| feat_3 | office |
| feat_4 | core |

## 1. Why `torch.no_grad()`?

During training, PyTorch builds a **computation graph** to track operations
so gradients can be back-propagated. At inference time this graph is wasteful:

| Without `no_grad` | With `torch.no_grad()` |
|---|---|
| Gradient tensors allocated | No gradient storage |
| Slower forward pass | Faster forward pass |
| Higher memory usage | Lower memory usage |

Combined with `model.eval()`, which disables **dropout** and switches
**batch normalisation** to use running statistics, these two calls guarantee
a pure, deterministic inference run.

## 2. Input Graph Format

Three CSV files are required in the same directory:

### `graphs.csv`
```
graph_id, label, feat
0, 4,
```

### `nodes.csv`
```
graph_id, node_id, label, feat_0, feat_1, feat_2, feat_3, feat_4
0, 0, 0, 1, 0, 0, 0, 0   ← ground cell
0, 1, 3, 0, 0, 0, 1, 0   ← office cell
...
```

### `edges.csv`
```
graph_id, src_id, dst_id, label, feat
0, 0, 20, 0,
0, 0, 13, 0,
...
```

The `label` column in `graphs.csv` is the manually-assigned BGR class
(your assessment of the building). The model's predicted class may or may
not agree with it.

## 3. Setup and Imports

In [ ]:
from pathlib import Path
import pandas as pd
import torch

# Paths (relative to repo root — adjust if running from a different location)
REPO_ROOT  = Path(".").resolve().parent
INPUT_DIR  = REPO_ROOT / "data" / "input"
MODEL_PATH = REPO_ROOT / "data" / "model" / "bgr_model.pt"
OUTPUT_PATH = REPO_ROOT / "data" / "predictions.csv"

CLASS_MAPPING = {
    0: "Separation",
    1: "Separation with Plinth",
    2: "Adherence",
    3: "Adherence with Plinth",
    4: "Interlock",
}

print(f"Input  : {INPUT_DIR}")
print(f"Model  : {MODEL_PATH}")
print(f"torch  : {torch.__version__}")

## 4. Load and Inspect the Input Graph

In [ ]:
graphs_df = pd.read_csv(INPUT_DIR / "graphs.csv")
nodes_df  = pd.read_csv(INPUT_DIR / "nodes.csv")
edges_df  = pd.read_csv(INPUT_DIR / "edges.csv")

print(f"Graphs : {len(graphs_df)} graph(s)")
print(f"Nodes  : {len(nodes_df)} node(s)")
print(f"Edges  : {len(edges_df)} edge(s)")
print()
print("graphs.csv:")
display(graphs_df.head())
print("nodes.csv (first 5 rows):")
display(nodes_df.head())

## 5. Verify Node Features

In [ ]:
REQUIRED = ["feat_0", "feat_1", "feat_2", "feat_3", "feat_4"]
CELL_TYPES = ["ground", "column", "plinth", "office", "core"]

missing = [f for f in REQUIRED if f not in nodes_df.columns]
assert not missing, f"Missing features: {missing}"

print("Feature columns present — node type distribution:")
for feat, name in zip(REQUIRED, CELL_TYPES):
    count = int(nodes_df[feat].sum())
    print(f"  {feat} ({name:8s}) : {count:3d} nodes")

## 6. Load Dataset and Model via topologicpy.PyG

In [ ]:
from topologicpy.PyG import PyG

# Build the PyG object — this parses the CSVs and infers input dimensions
pyg_eval = PyG.ByCSVPath(
    path=str(INPUT_DIR),
    level="graph",
    task="classification",
    graphLabelType="categorical",
    nodeLabelType="categorical",
    edgeLabelType="categorical",
)

# Load the pre-trained weights into the constructed model
pyg_eval.LoadModel(str(MODEL_PATH))
print("Model loaded from:", MODEL_PATH)

## 7. Set `model.eval()` and Run with `torch.no_grad()`

In [ ]:
# Route all graphs to the test split (0% train, 0% val, 100% test)
pyg_eval.SetHyperparameters(split=(0.0, 0.0, 1.0), shuffle=False)

# Explicitly set evaluation mode
# — disables dropout, uses running stats for batch normalisation
if pyg_eval.model is not None:
    pyg_eval.model.eval()
    print("model.eval() called")

# Run forward pass without computing gradients
print("Running inference inside torch.no_grad() ...")
with torch.no_grad():
    pred_results = pyg_eval.Predict()

print("Inference complete. No gradients were computed.")

## 8. Decode and Display Results

In [ ]:
predictions   = pred_results["pred"].tolist()
probabilities = pred_results["prob"].tolist()

results_df = pd.DataFrame({
    "graph_id":        range(len(predictions)),
    "predicted_class": predictions,
    "predicted_label": [CLASS_MAPPING[int(p)] for p in predictions],
    "confidence":      [round(max(p), 4) for p in probabilities],
})

results_df.to_csv(OUTPUT_PATH, index=False)
print(f"Saved to: {OUTPUT_PATH}")
display(results_df)

## 9. Visualise the Workflow

In [ ]:
from IPython.display import Image, display as ipy_display

visuals = REPO_ROOT / "visuals"
for name in ["evaluation_workflow.png", "input_graph.png", "prediction_result.png"]:
    print(f"\n--- {name} ---")
    ipy_display(Image(str(visuals / name), width=800))

## 10. Limitations

| Limitation | Detail |
|------------|--------|
| **Single graph evaluation** | Only one graph (graph_id 0) is included in `data/input`. Results are not statistically representative. |
| **No ground-truth validation** | The `label` in `graphs.csv` is a manual assessment. Model agreement is illustrative, not a metric. |
| **Topologicpy dependency** | The `PyG` wrapper requires `topologicpy >= 0.9.31`, `torch`, `torch_geometric`, and related packages. |
| **Model architecture opaque** | `bgr_model.pt` was trained by the course instructor. The exact architecture, training data split, and performance metrics are not accessible here. |
| **No GPU assumed** | The script runs on CPU. Inference is fast for a single graph but may differ from GPU behaviour with batch normalisation. |
| **Confidence is softmax, not calibrated** | The confidence score is a raw softmax probability, not a calibrated uncertainty estimate. |
| **Prediction result is inferred in visuals** | The bar chart in `prediction_result.png` uses an estimated confidence distribution `[inferred]` since the script was authored without live model execution in this environment. |

## Possible Improvements

- Evaluate on multiple buildings to get a distribution of predictions.
- Compare model prediction with a manual label for each building.
- Add SHAP or attention-weight visualisation to explain which nodes drive the prediction.
- Package the evaluation into a command-line tool with `argparse` for batch runs.